In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [1]:
import pandas as pd
import scipy.sparse as sp
from sklearn.linear_model import LogisticRegression
import pickle

print("Libraries loaded successfully")

Libraries loaded successfully


In [4]:
# Load training set
X_train = sp.load_npz("/content/drive/MyDrive/FactLens_Group9/data/X_train.npz")
y_train = pd.read_csv("/content/drive/MyDrive/FactLens_Group9/data/y_train.csv")["label"]

print(f"Training matrix loaded: {X_train.shape}")
print(f"Training labels loaded: {len(y_train)}")
print(f"\nLabel distribution:")
print(y_train.value_counts())

Training matrix loaded: (30872, 50000)
Training labels loaded: 30872

Label distribution:
label
REAL    16953
FAKE    13919
Name: count, dtype: int64


In [5]:
print("Training Logistic Regression model...")
print("This will take 2-3 minutes...")

model = LogisticRegression(
    C=1.0,                    # regularization strength - controls overfitting
    max_iter=1000,            # maximum number of iterations to converge
    random_state=42,          # reproducibility
    class_weight="balanced",  # handles our 54/46 class imbalance
    solver="lbfgs",           # optimization algorithm - best for text data
    n_jobs=-1                 # use all available CPU cores to speed up training
)

model.fit(X_train, y_train)

print("Training complete")

Training Logistic Regression model...
This will take 2-3 minutes...
Training complete


In [6]:
# Test on a few training examples to make sure model works
sample_preds = model.predict(X_train[:5])
sample_probs = model.predict_proba(X_train[:5])

print("Sample predictions on first 5 training articles:")
for i in range(5):
    actual = y_train.iloc[i]
    predicted = sample_preds[i]
    fake_prob = sample_probs[i][0]
    real_prob = sample_probs[i][1]
    correct = "✓" if actual == predicted else "✗"
    print(f"  Article {i+1}: Actual={actual:4} | Predicted={predicted:4} | Fake prob={fake_prob:.2f} | Real prob={real_prob:.2f} {correct}")

Sample predictions on first 5 training articles:
  Article 1: Actual=FAKE | Predicted=FAKE | Fake prob=0.97 | Real prob=0.03 ✓
  Article 2: Actual=FAKE | Predicted=FAKE | Fake prob=0.92 | Real prob=0.08 ✓
  Article 3: Actual=REAL | Predicted=REAL | Fake prob=0.06 | Real prob=0.94 ✓
  Article 4: Actual=FAKE | Predicted=FAKE | Fake prob=1.00 | Real prob=0.00 ✓
  Article 5: Actual=FAKE | Predicted=FAKE | Fake prob=1.00 | Real prob=0.00 ✓


In [7]:
# Get all predictions on training set
train_preds = model.predict(X_train)
train_probs = model.predict_proba(X_train)

# How confident is the model on average?
import numpy as np
max_probs = train_probs.max(axis=1)

print(f"Average model confidence on training set: {max_probs.mean():.2%}")
print(f"Predictions above 90% confidence:         {(max_probs > 0.9).sum():,}")
print(f"Predictions above 70% confidence:         {(max_probs > 0.7).sum():,}")
print(f"Predictions below 60% confidence:         {(max_probs < 0.6).sum():,} (uncertain predictions)")

Average model confidence on training set: 93.43%
Predictions above 90% confidence:         24,756
Predictions above 70% confidence:         29,983
Predictions below 60% confidence:         347 (uncertain predictions)


In [8]:
with open("/content/drive/MyDrive/FactLens_Group9/data/logistic_regression_model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved successfully as logistic_regression_model.pkl")

Model saved successfully as logistic_regression_model.pkl
